# Libraries

In [1]:
# 0. Clean slate — remove existing copies before reinstalling
!pip uninstall -y torch torchvision torchaudio torchao

!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 torchao==0.16.0 --index-url https://download.pytorch.org/whl/cu128

!pip install -q "cuda-core==0.3.*" "numba-cuda[cu12]<0.23.0,>=0.22.2" "numba<0.62.0,>=0.60.0"

!pip install -q llmcompressor==0.10.0.2

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 3.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.7 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 14.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.9 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver 

In [2]:
import random
import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)

from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier

from huggingface_hub import (
    login,
    create_repo,
    upload_folder,
)

from kaggle_secrets import UserSecretsClient
from datasets import load_dataset, Dataset
import json

2026-06-07 10:26:19.913372: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780827980.114974      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780827980.170627      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780827980.648947      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780827980.648985      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780827980.648988      58 computation_placer.cc:177] computation placer alr

# Global Variables

In [3]:
MODEL_ID = "Qwen/Qwen3-4B"
OUTPUT_DIR = "/kaggle/working/Qwen3-4B-AWQ-4bit"
REPO_ID = f"EdgeCompress01/Qwen3-4B-AWQ-4bit-2026"
SEED = 42

# Functions

In [4]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")


def format_chat(batch):
    texts = [
        tokenizer.apply_chat_template(
            text,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )
        for text in batch["text"]
    ]
    return {"text": texts}

# Set Seed

In [5]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface Login

In [6]:
# --- 3. HUGGING FACE AUTHENTICATION ---
print("Logging into Hugging Face...")
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

Logging into Hugging Face...


# Load Model & Tokenizer

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype="bfloat16"
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

# AWQ Quantization

**Configurations**

In [8]:
recipe = [
    AWQModifier(
        ignore=["lm_head", "embed_tokens"], 
        scheme="W4A16",
        targets=["Linear"]
    )
]

**Calibration Dataset**

In [9]:
dataset = load_dataset("monology/pile-uncopyrighted-parquet", split="train", streaming=True).shuffle(seed=SEED, buffer_size=1000)

samples = Dataset.from_dict({"text": [[{"role": "user", "content": s["text"]}] for s in dataset.take(64)]})

formatted_dataset = samples.map(format_chat, batched=True)

README.md:   0%|          | 0.00/367 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

**Apply quantization**

In [10]:
oneshot(
    model=model,
    tokenizer=tokenizer,
    dataset=formatted_dataset,
    text_column="text",
    recipe=recipe,
    max_seq_length=2048,
    num_calibration_samples=64
)

2026-06-07T10:27:40.252849+0000 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.


Tokenizing:   0%|          | 0/64 [00:00<?, ? examples/s]

2026-06-07T10:27:40.778451+0000 | reset | INFO - Compression lifecycle reset
2026-06-07T10:27:40.781363+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-06-07T10:27:40.852226+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...
2026-06-07T10:27:40.865185+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.0.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-06-07T10:27:40.866253+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.1.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-06-07T10:27:40.867741+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.2.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', b

W0607 10:27:40.957000 58 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
Smoothing:  33%|███▎      | 1/3 [01:57<03:55, 117.96s/it]                                                            
Grid search for model.layers.0.post_attention_layernorm:   0%|          | 0/20 [00:00<?, ?it/s]
Grid search for model.layers.0.post_attention_layernorm:   0%|          | 0/20 [00:09<?, ?it/s, best_error=2.742e-05]
Grid search for model.layers.0.post_attention_layernorm:   5%|▌         | 1/20 [00:09<02:55,  9.24s/it, best_error=2.742e-05]
Grid search for model.layers.0.post_attention_layernorm:   5%|▌         | 1/20 [00:18<02:55,  9.24s/it, best_error=2.694e-05]
Grid search for model.layers.0.post_attention_layernorm:  10%|█         | 2/20 [00:18<02:46,  9.23s/it, best_error=2.694e-05]
Grid s

2026-06-07T14:41:13.889849+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


2026-06-07T14:41:13.890429+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [11]:
model.config._name_or_path = ""

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved successfully")

2026-06-07T14:41:13.905903+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 252it [00:33,  7.60it/s]
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3970: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully


# PUSH TO HUGGING FACE

In [12]:
# create repo in organization
create_repo(REPO_ID, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=OUTPUT_DIR
)

print("Upload complete to organization!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete to organization!
